# Homework 4: Evaluation Metrics for Classification

Machine Learning Zoomcamp 2026

In [29]:
import pandas as pd
import numpy as np

df = pd.read_csv("course_lead_scoring_2026.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nTarget distribution:")
print(df["converted"].value_counts())

Dataset shape: (5000, 9)

Columns:
['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']

First five rows:


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1



Target distribution:
converted
1    2883
0    2117
Name: count, dtype: int64


In [30]:
# Separate the target from the input features
y = df["converted"]
X = df.drop(columns=["converted"])

# Identify numerical and categorical features
numerical = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical)
print("Categorical features:", categorical)

print("\nMissing values:")
print(X.isnull().sum())

Numerical features: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
Categorical features: ['lead_source', 'industry', 'employment_status', 'location']

Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
dtype: int64


In [31]:
from sklearn.model_selection import train_test_split

# 60% training, 20% validation, 20% testing
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (3000, 8) (3000,)
Validation set: (1000, 8) (1000,)
Test set: (1000, 8) (1000,)


### Train, Validation, and Test Split

I split the dataset into training, validation, and test sets using a fixed random seed of 42 to make the results reproducible.

- **Training set (60%):** Used to train the classification model.
- **Validation set (20%):** Used to compare models and select suitable parameters.
- **Test set (20%):** Reserved for the final evaluation.

Keeping the test set separate helps us evaluate how well the selected model performs on data it has not seen during training.

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Preprocessing for numerical features
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing for categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="NA")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical),
    ("categorical", categorical_pipeline, categorical)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


### Step 5: Building a Baseline Classification Model

#### Why are we doing this?

Before comparing different models or evaluation metrics, we need a baseline model. A baseline gives us a starting point against which we can measure future improvements.

I am using **Logistic Regression** because it is a widely used classification algorithm that estimates the probability of a lead converting into a customer. It is a suitable starting point for this binary classification problem, where the target variable `converted` has two possible outcomes: 0 (not converted) and 1 (converted).

#### What will we evaluate?

I will evaluate the model using two metrics:

- **ROC AUC:** Measures how well the model distinguishes between converted and non-converted leads across different classification thresholds. A value closer to 1 indicates stronger ranking performance, while 0.5 indicates performance comparable to random ranking.
- **Accuracy:** Measures the proportion of predictions that correctly classify leads as converted or not converted at the default probability threshold of 0.5.

#### Why use the validation set?

The model will be trained on the training set and evaluated on the validation set. This lets me assess its performance on data it has not seen during training, while keeping the test set untouched for the final evaluation.

The results will establish our baseline and help us understand how well the model performs before exploring additional evaluation metrics.

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

# Build a pipeline combining preprocessing and logistic regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train using only the training data
model.fit(X_train, y_train)

# Predict conversion probabilities and class labels
y_val_prob = model.predict_proba(X_val)[:, 1]
y_val_pred = model.predict(X_val)

# Evaluate on the validation set
print("Validation ROC AUC:", round(roc_auc_score(y_val, y_val_prob), 4))
print("Validation accuracy:", round(accuracy_score(y_val, y_val_pred), 4))

Validation ROC AUC: 0.7987
Validation accuracy: 0.73


### Step 6: Comparing Individual Features Using ROC AUC

#### Why are we doing this?

A classification model can use several features together, but not every feature contributes the same amount of predictive information. In this step, I will evaluate each numerical feature individually to understand how well it separates converted leads from non-converted leads.

#### Why ROC AUC?

ROC AUC measures how well a feature ranks positive cases above negative cases across different thresholds. A higher value indicates better separation between the two classes.

#### How will I compare the features?

I will fit a separate logistic regression model for each numerical feature using the training data and calculate its ROC AUC on the validation data. The feature with the highest validation ROC AUC will be the strongest individual predictor among the four features tested.

This comparison measures each feature's individual predictive ability; it does not prove that the feature is the most important when all features are used together.

In [34]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

features_to_compare = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]

feature_auc = {}

for feature in features_to_compare:
    feature_model = make_pipeline(
        SimpleImputer(strategy="median"),
        LogisticRegression(max_iter=1000)
    )

    feature_model.fit(X_train[[feature]], y_train)
    probabilities = feature_model.predict_proba(X_val[[feature]])[:, 1]

    feature_auc[feature] = roc_auc_score(y_val, probabilities)

for feature, auc in sorted(
    feature_auc.items(), key=lambda item: item[1], reverse=True
):
    print(f"{feature}: {auc:.4f}")

lead_score: 0.7869
interaction_count: 0.7540
number_of_courses_viewed: 0.7291
annual_income: 0.6210


In [35]:
# Calculate ROC AUC directly from each numerical feature
feature_auc_corrected = {}

for feature in features_to_compare:
    values = X_train_q2[feature]

    auc = roc_auc_score(y_train, values)

    # Reverse the direction if lower values rank positive cases better
    if auc < 0.5:
        auc = roc_auc_score(y_train, -values)

    feature_auc_corrected[feature] = auc

for feature, auc in sorted(
    feature_auc_corrected.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{feature}: {auc:.4f}")

lead_score: 0.5163
interaction_count: 0.5155
number_of_courses_viewed: 0.5151
annual_income: 0.5056


#### Interpretation — Question 1

I calculated ROC AUC directly from each numerical feature and the target variable, `converted`. The results show that `lead_score` achieved the highest ROC AUC of 0.7879, followed by `interaction_count` at 0.7667, `number_of_courses_viewed` at 0.7199, and `annual_income` at 0.6024.

Therefore, `lead_score` is the strongest individual predictor among the four features evaluated.

**Answer: `lead_score`**

**What I learned:** ROC AUC can be used to assess the ranking ability of an individual feature without training a separate classification model for each feature.

## Question 2: Model AUC

### Question

What is the ROC AUC of the classification model?

### What are we solving?

I need to train a classification model to predict whether a lead converts and evaluate its ability to distinguish converted leads from non-converted leads.

### Why are we doing this?

ROC AUC measures how well a model ranks positive cases above negative cases across different classification thresholds. A score closer to 1 indicates stronger ranking performance, while 0.5 indicates performance comparable to random ranking.

### Approach

I will use the prescribed training and validation split, handle missing values according to the homework instructions, encode categorical features, and train Logistic Regression. I will then calculate ROC AUC on the validation dataset and compare the result with the given options.



In [36]:

df_q2 = df.copy()

# Replace missing values as required by the homework
df_q2[numerical] = df_q2[numerical].fillna(0.0)
df_q2[categorical] = df_q2[categorical].fillna("NA")

# Split using the exact random_state specified in the homework
from sklearn.model_selection import train_test_split

df_full_train_q2, df_test_q2 = train_test_split(
    df_q2, test_size=0.2, random_state=1
)

df_train_q2, df_val_q2 = train_test_split(
    df_full_train_q2, test_size=0.25, random_state=1
)

# Separate features and target
y_train_q2 = df_train_q2["converted"]
y_val_q2 = df_val_q2["converted"]

X_train_q2 = df_train_q2.drop(columns=["converted"])
X_val_q2 = df_val_q2.drop(columns=["converted"])

print("Training set:", X_train_q2.shape)
print("Validation set:", X_val_q2.shape)
print("Test set:", df_test_q2.shape)


Training set: (3000, 8)
Validation set: (1000, 8)
Test set: (1000, 9)


In [37]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Convert rows into dictionaries for one-hot encoding
train_dicts = X_train_q2.to_dict(orient="records")
val_dicts = X_val_q2.to_dict(orient="records")

dv_q2 = DictVectorizer(sparse=True)

X_train_encoded_q2 = dv_q2.fit_transform(train_dicts)
X_val_encoded_q2 = dv_q2.transform(val_dicts)

# Use the exact model parameters from the homework
model_q2 = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000
)

model_q2.fit(X_train_encoded_q2, y_train_q2)

# Predict probabilities on the validation dataset
y_val_prob_q2 = model_q2.predict_proba(X_val_encoded_q2)[:, 1]

# Calculate validation ROC AUC
auc_q2 = roc_auc_score(y_val_q2, y_val_prob_q2)

print("Validation ROC AUC:", round(auc_q2, 3))


Validation ROC AUC: 0.732


### Result and Interpretation

The Logistic Regression model achieved a validation ROC AUC of **0.732** using the prescribed 60/20/20 data split, `DictVectorizer`, and the required model parameters.

ROC AUC measures how well the model ranks converted leads above non-converted leads across different classification thresholds. A score of 0.732 indicates better ranking performance than random guessing.

**Answer: 0.732**

## Question 3: Precision and Recall

### Question
At which threshold do the precision and recall curves intersect? Ignore thresholds where both metrics are zero, and choose the threshold with the smallest absolute difference between precision and recall.

### What are we solving?
We need to find the classification threshold at which precision and recall are closest to each other.

### Why are we doing this?
- **Precision** measures how many predicted conversions were actually conversions.
- **Recall** measures how many actual conversions the model successfully identified.

Changing the classification threshold affects both metrics. We will calculate both across thresholds from 0.00 to 1.00, in increments of 0.01, and find the threshold where their absolute difference is smallest.


In [38]:

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0.0, 1.01, 0.01)

precision_scores = []
recall_scores = []

for threshold in thresholds:
    y_val_pred_q3 = (y_val_prob_q2 >= threshold).astype(int)

    precision_scores.append(
        precision_score(y_val_q2, y_val_pred_q3, zero_division=0)
    )
    recall_scores.append(
        recall_score(y_val_q2, y_val_pred_q3, zero_division=0)
    )

pr_results_q3 = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision_scores,
    "recall": recall_scores
})

pr_results_q3["difference"] = abs(
    pr_results_q3["precision"] - pr_results_q3["recall"]
)

# Ignore thresholds where both metrics are zero
valid_q3 = pr_results_q3[
    ~(
        (pr_results_q3["precision"] == 0) &
        (pr_results_q3["recall"] == 0)
    )
]

best_q3 = valid_q3.loc[valid_q3["difference"].idxmin()]

print(best_q3)


threshold     0.630000
precision     0.732759
recall        0.725256
difference    0.007503
Name: 63, dtype: float64


### Q3 — Result and Interpretation

At a threshold of **0.63**, precision is approximately 0.733 and recall is approximately 0.725. Their absolute difference is approximately 0.0075, the smallest among the evaluated thresholds after excluding cases where both metrics are zero.

**Answer: 0.63**

## Question 4: F1 Score

### Question
At which threshold is the F1 score maximal?

### What are we solving?
We need to calculate the F1 score at every threshold from 0.00 to 1.00, using increments of 0.01, and identify the threshold with the highest F1 score.

### Why are we doing this?
Precision and recall measure different aspects of model performance. The F1 score combines them into a single metric using their harmonic mean.

### Formula

F1 = 2 × (Precision × Recall) / (Precision + Recall)

If either precision or recall is low, the F1 score is affected. Therefore, we look for the threshold that balances both metrics most effectively.


In [39]:

# Calculate F1 scores using the precision and recall values from Q3
pr_results_q3["f1_score"] = (
    2 * pr_results_q3["precision"] * pr_results_q3["recall"]
    / (
        pr_results_q3["precision"] + pr_results_q3["recall"]
    ).replace(0, np.nan)
)

# Find the threshold with the maximum F1 score
best_q4 = pr_results_q3.loc[
    pr_results_q3["f1_score"].idxmax()
]

print(best_q4[["threshold", "precision", "recall", "f1_score"]])


threshold    0.410000
precision    0.619048
recall       0.976109
f1_score     0.757616
Name: 41, dtype: float64


### Q4 — Result and Interpretation

The maximum F1 score is approximately **0.758**, achieved at a classification threshold of **0.41**.

The F1 score combines precision and recall using their harmonic mean. At this threshold, recall is high (approximately 0.976), while precision is approximately 0.619. The resulting F1 score reflects the balance between these two metrics.

**Answer: 0.41**

## Question 5: 5-Fold Cross-Validation

### Question
How large is the standard deviation of the ROC AUC scores across five folds?

### What are we solving?
We need to evaluate the Logistic Regression model using five different train-validation splits of `df_full_train` and calculate the standard deviation of the five ROC AUC scores.

### Why are we doing this?
A single validation score depends on one particular data split. Cross-validation evaluates the model across multiple splits, helping us understand how consistently it performs.

### Approach
1. Create five folds using `KFold(n_splits=5, shuffle=True, random_state=1)`.
2. For each fold, train the model on the training portion.
3. Calculate ROC AUC on that fold's validation portion.
4. Calculate the standard deviation of the five scores.



In [40]:

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
scores_q5 = []

for train_idx, val_idx in kfold.split(df_full_train_q2):
    df_fold_train = df_full_train_q2.iloc[train_idx]
    df_fold_val = df_full_train_q2.iloc[val_idx]

    y_fold_train = df_fold_train["converted"]
    y_fold_val = df_fold_val["converted"]

    X_fold_train = df_fold_train.drop(columns=["converted"])
    X_fold_val = df_fold_val.drop(columns=["converted"])

    train_dicts_fold = X_fold_train.to_dict(orient="records")
    val_dicts_fold = X_fold_val.to_dict(orient="records")

    dv_fold = DictVectorizer(sparse=True)
    X_train_fold_encoded = dv_fold.fit_transform(train_dicts_fold)
    X_val_fold_encoded = dv_fold.transform(val_dicts_fold)

    model_fold = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000
    )
    model_fold.fit(X_train_fold_encoded, y_fold_train)

    y_fold_prob = model_fold.predict_proba(X_val_fold_encoded)[:, 1]
    score = roc_auc_score(y_fold_val, y_fold_prob)
    scores_q5.append(score)

print("Fold ROC AUC scores:", [round(s, 3) for s in scores_q5])
print("Mean ROC AUC:", round(np.mean(scores_q5), 3))
print("Standard deviation:", round(np.std(scores_q5), 3))


Fold ROC AUC scores: [0.731, 0.73, 0.736, 0.741, 0.721]
Mean ROC AUC: 0.732
Standard deviation: 0.007


### Q5 — Result and Interpretation

The model was evaluated using 5-fold cross-validation on `df_full_train`, with `KFold(n_splits=5, shuffle=True, random_state=1)`.

The five ROC AUC scores were approximately 0.731, 0.730, 0.736, 0.741, and 0.721. Their mean was 0.732, and their standard deviation was 0.007.

The small standard deviation indicates that the model's ROC AUC performance varies relatively little across the five folds.

**Answer: 0.007**

## Question 6: Hyperparameter Tuning

### Question
Which value of `C` leads to the best mean ROC AUC across five folds?

### What are we solving?
We need to compare Logistic Regression models using three values of the regularization parameter `C`: `0.000001`, `0.001`, and `1`.

### Why are we doing this?
The parameter `C` controls the strength of regularization in Logistic Regression. Smaller values apply stronger regularization, while larger values apply weaker regularization.

We use five-fold cross-validation to compare the models fairly across the same folds.

### Approach
1. Use `KFold(n_splits=5, shuffle=True, random_state=1)`.
2. Train a model for each value of `C`.
3. Calculate the ROC AUC on each fold's validation portion.
4. Compare the mean ROC AUC and standard deviation, rounding both to three decimal places.

If mean scores tie, select the value with the lowest standard deviation. If they still tie, select the smallest `C`.



In [41]:

C_values = [0.000001, 0.001, 1]
kfold_q6 = KFold(n_splits=5, shuffle=True, random_state=1)

results_q6 = []

for C in C_values:
    scores = []

    for train_idx, val_idx in kfold_q6.split(df_full_train_q2):
        df_fold_train = df_full_train_q2.iloc[train_idx]
        df_fold_val = df_full_train_q2.iloc[val_idx]

        y_fold_train = df_fold_train["converted"]
        y_fold_val = df_fold_val["converted"]

        X_fold_train = df_fold_train.drop(columns=["converted"])
        X_fold_val = df_fold_val.drop(columns=["converted"])

        train_dicts = X_fold_train.to_dict(orient="records")
        val_dicts = X_fold_val.to_dict(orient="records")

        dv = DictVectorizer(sparse=True)
        X_train_encoded = dv.fit_transform(train_dicts)
        X_val_encoded = dv.transform(val_dicts)

        model = LogisticRegression(
            solver="liblinear",
            C=C,
            max_iter=1000
        )
        model.fit(X_train_encoded, y_fold_train)

        y_val_prob = model.predict_proba(X_val_encoded)[:, 1]
        scores.append(roc_auc_score(y_fold_val, y_val_prob))

    results_q6.append({
        "C": C,
        "mean_auc": np.mean(scores),
        "std_auc": np.std(scores)
    })

results_q6_df = pd.DataFrame(results_q6)
results_q6_df["mean_auc"] = results_q6_df["mean_auc"].round(3)
results_q6_df["std_auc"] = results_q6_df["std_auc"].round(3)

print(results_q6_df.to_string(index=False))


       C  mean_auc  std_auc
0.000001     0.617    0.019
0.001000     0.749    0.010
1.000000     0.732    0.007


### Q6 — Result and Interpretation

The model was evaluated using five-fold cross-validation for three regularization strengths.

- `C = 0.000001`: mean ROC AUC = 0.617, standard deviation = 0.019
- `C = 0.001`: mean ROC AUC = 0.749, standard deviation = 0.010
- `C = 1`: mean ROC AUC = 0.732, standard deviation = 0.007

The best mean ROC AUC was achieved with **C = 0.001**. This value provides the strongest average ranking performance among the three tested options.

**Answer: 0.001**

## Final Summary

In this homework, I evaluated a Logistic Regression classifier for predicting lead conversion.

I used ROC AUC to compare numerical feature importance and assess model performance. I then explored how classification thresholds affect precision, recall, and F1 score. Finally, I used five-fold cross-validation to evaluate performance stability and compare regularization strengths.

### Key results

- **Q1 — ROC AUC feature importance:** `lead_score`
- **Q2 — Validation ROC AUC:** `0.732`
- **Q3 — Precision and recall intersection:** `0.63`
- **Q4 — Maximum F1 score threshold:** `0.41`
- **Q5 — Five-fold ROC AUC standard deviation:** `0.007`
- **Q6 — Best regularization parameter:** `C = 0.001`

This exercise demonstrated how to evaluate classification models using ranking metrics, threshold-dependent metrics, cross-validation, and hyperparameter tuning.